# A small network for ice cream sales

A stall sells more ice cream the warmer the day, along an S-shaped curve. A small **neural network** learns this curve from $300$ days: it reads the temperature of a day and predicts how many ice creams the stall sells.

The days are synthetic, made up: a few lines of Python below generate them with a seeded random generator, so every run sees the same days.

In [ ]:
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt

## 1. Made-up data

Each day has two numbers, both between $0$ and $1$, a comfortable size for a network:

* the temperature, from $0$ (a cold day of $5$ degrees) to $1$ (a hot day of $35$ degrees),
* the ice creams sold, as a share of the stall's stock, from $0$ (nothing sold) to $1$ (sold out).

Cold days sell almost nothing, around $20$ degrees the sales climb steeply, and hot days sell out. Luck adds some noise to every day.

In [ ]:
rng = np.random.default_rng(seed=0)   # seeded: the same 300 days on every computer
n = 300                               # the number of days

temperature = rng.uniform(low=0, high=1, size=n)   # one temperature per day, between 0 and 1

# The average sales on a day with this temperature: an S shape, near 0 when it is cold,
# near 1 when it is hot and steepest at 0.5.
average_sales = 1 / (1 + np.exp(-10 * (temperature - 0.5)))
luck = rng.normal(loc=0, scale=0.05, size=n)   # the noise: a bit above or below each day
sales = np.clip(average_sales + luck, 0, 1)    # between 0 (nothing sold) and 1 (sold out)

plt.scatter(temperature, sales, s=12, color="#0072B2")
plt.xlabel("temperature (0 = 5 degrees, 1 = 35 degrees)")
plt.ylabel("ice creams sold (1 = sold out)")
plt.show()

In [ ]:
# A network reads a table with one row per day and one column per number,
# so each list of n numbers becomes a column of shape (n, 1): n rows, 1 column.
# float32 is the kind of number a torch network works with.
x = torch.tensor(temperature, dtype=torch.float32).reshape(n, 1)   # what the network reads
y = torch.tensor(sales, dtype=torch.float32).reshape(n, 1)         # what it should predict
print(x.shape, y.shape)

## 2. The model

The network reads one number, the temperature, and gives back one number, its prediction for the ice creams sold. In between sit two **hidden layers** of $32$ numbers each. Each hidden layer is one line of the code below: an `nn.Linear`, which computes with weights that training adjusts, followed by an `nn.ReLU`, which turns every negative number into $0$. These bends let the network follow a curve.

In [ ]:
torch.manual_seed(0)   # seeded: the same starting weights on every computer

# One line is one layer: an nn.Linear, then an nn.ReLU. nn.Linear(a, b) takes a numbers in
# and gives b numbers out, so each line takes in as many numbers as the line above gives out.
# The last line has no nn.ReLU: its numbers are the answer.
model = nn.Sequential(
    nn.Linear(1, 32), nn.ReLU(),     # 1 number in (the temperature), 32 numbers out
    nn.Linear(32, 32), nn.ReLU(),    # 32 numbers in (from the line above), 32 numbers out
    nn.Linear(32, 1),                # 32 numbers in, 1 number out: the prediction
)

## 3. Training

The loop of weekend 1: while the miss is too big, take a step. The miss is the squared difference between the predicted and the true ice creams sold, averaged over all $300$ days. The days are noisy, so no smooth curve can bring the average squared miss down to $0$. The loop stops at $0.003$, which is close to what the noise alone causes. With more noise in the days, raise this number with it: a miss below the noise level takes very long to reach.

In [ ]:
# An optimizer changes the weights of the model a little at every step, to shrink the miss.
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)   # lr: how big a step is

loss = 1.0                                   # any start above the threshold
while loss > 0.003:
    loss = ((model(x) - y) ** 2).mean()      # the average squared miss
    optimizer.zero_grad()                    # forget the gradients of the previous step
    loss.backward()                          # compute the gradient of the loss
    optimizer.step()                         # move every weight a small step against its gradient
    print(f"\raverage squared miss: {loss.item():.4f}", end="")

In [ ]:
# The prediction for 100 temperatures from 0 to 1, laid out as a column like x.
# detach() drops the bookkeeping of training, so matplotlib can draw the numbers.
grid = torch.linspace(0, 1, 100).reshape(100, 1)
curve = model(grid).detach()

plt.scatter(temperature, sales, s=12, color="#0072B2", label="300 made-up days")
plt.plot(grid, curve, color="#D55E00", linewidth=2.5, label="the network")
plt.xlabel("temperature (0 = 5 degrees, 1 = 35 degrees)")
plt.ylabel("ice creams sold (1 = sold out)")
plt.legend()
plt.show()

## 4. What the network learned

The dots are the $300$ made-up days and the curve is the network's prediction for every temperature from $0$ to $1$. Nobody told the network the rule that made the days: it found the S shape from the dots alone.

## The three patterns in words

* **A layer** is one line inside `nn.Sequential`: `nn.Linear(a, b), nn.ReLU(),`. The number `a` says how many numbers come in, the number `b` how many come out, and the `b` of one line is the `a` of the next. The last line has no `nn.ReLU()`.
* **The optimizer** is created once, before the loop, from the weights of the model and the size of a step: `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`.
* **A training step** is three calls inside the loop, always in this order: `optimizer.zero_grad()` forgets the old gradients, `loss.backward()` computes the new ones, and `optimizer.step()` moves the weights.